# 01 - The RAG pipeline, one stage at a time

This notebook builds the whole Phase 1 pipeline on **1,000 auto-parts product listings**,
stopping at each stage so you can see what actually changed. Nothing is hidden behind a
framework - every function comes from `src/fitment_rag/`, so what you learn here is exactly what
runs in the sweeps.

```
  Amazon Automotive          [1] CORPUS       1,000 product listings
  product metadata     ->    load_documents()
  (HuggingFace)                   |
                                  v
                             [2] CHUNK        ~3,900 chunks of 512 chars
                             chunk_documents()
                                  |
                                  v
                             [3] EMBED        ~3,900 x 384 float32 matrix
                             Embedder.encode()
                                  |
                                  v
                             [4] INDEX        FAISS flat - exact, the ceiling
                             FaissStore.build()
                                  |
     question  ------------> [5] RETRIEVE     top-5 chunks
                             Retriever.retrieve()
                                  |
                                  v
                             [6] SCORE        recall@k, MRR, nDCG
                             score_query()
```

**Phase 1 stops at retrieval.** There is no generation step, and that is deliberate: an LLM
cannot rescue a chunk that was never retrieved, so measuring it before retrieval is settled
adds hours of CPU time and noise for no signal. Generation is Phase 3 - see
`context/02-plan.md`.

**Why the order matters:** stages 1-4 decide what *can* be found, and stage 5 decides what *is*
found. Most "the LLM hallucinated" complaints are really stage 2-5 failures.

## Setup

One cell, then never again. `%autoreload` means edits under `src/` take effect here without
restarting the kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

from fitment_rag.config import RunConfig

cfg = RunConfig.load("../configs/phase1_smoke.yaml")
print(f"config    : {cfg.name}    run_id: {cfg.run_id}")
print(f"corpus    : {cfg.data.n_docs} docs (stride {cfg.data.stride})")
print(f"chunking  : {cfg.chunking.strategy} {cfg.chunking.chunk_size}/{cfg.chunking.chunk_overlap}")
print(f"embed     : {cfg.embedding.model}")
print(f"index     : {cfg.vectorstore.backend}")
print(f"retrieval : {cfg.retrieval.mode}  top_k={cfg.retrieval.top_k}")

## Stage 1 - Corpus

`load_documents` streams the Amazon Reviews 2023 *Automotive product metadata* from HuggingFace
and keeps a deterministic sample.

Deterministic matters: rows are selected by a stable hash of each product's ASIN, **not** by
shuffling, so `n_docs=1000` gives the same 1,000 products on your laptop and on a reviewer's.

The first run downloads and caches into `data/raw/` (gitignored - we never redistribute the
dataset, we regenerate it). Expect a few minutes; every later run is instant.

In [ ]:
from fitment_rag.data.amazon import load_documents, corpus_checksum

docs = load_documents(cfg.data)
print(f"{len(docs)} documents")
print(f"checksum: {corpus_checksum(docs)}   <- quote this in your writeup")

print("\n--- one document ---")
print(docs[0]["text"][:600])

**Notice:** a document is a *flattened* listing - title, brand, categories, features,
description, structured details, all as one text blob. That flattening lives in
`data/amazon.py::_doc_text` and it is a real design decision: it is the reason a question about
price is answerable by retrieval at all. Change the flattening and you change your ceiling.

## Stage 2 - Chunking

Embedding models truncate (MiniLM stops at 256 word-pieces), so a 3,000-character listing has to
be split. The split decides what is findable: too large and the matching sentence gets diluted by
everything around it, too small and the answer gets separated from the title that identifies it.

Chunk ids are `{doc_id}::{ordinal}`, so every hit traces back to a source document. That is what
lets us score retrieval at the *document* level and keep chunk sizes comparable.

In [ ]:
from fitment_rag.chunking import chunk_documents

chunks = chunk_documents(docs, cfg.chunking)
print(f"{len(docs)} docs -> {len(chunks)} chunks  "
      f"({len(chunks)/len(docs):.1f} per doc)")

for c in chunks[:2]:
    print(f"\n[{c['chunk_id']}]  {len(c['text'])} chars")
    print(c["text"][:200], "...")

In [ ]:
# Preview of Phase 1b: how much does the strategy change the corpus?
from fitment_rag.config import ChunkConfig

for strat, size in [("whole_doc", 512), ("fixed", 256), ("fixed", 512),
                    ("fixed", 1024), ("sentence", 512)]:
    n = len(chunk_documents(docs, ChunkConfig(strategy=strat, chunk_size=size,
                                              chunk_overlap=size // 8)))
    print(f"{strat:10} size={size:<6} ->  {n:7,} chunks")

## Stage 3 - Embedding

`all-MiniLM-L6-v2` is 22M parameters mapping text into 384 dimensions. On your CPU that runs at
roughly 200-400 chunks/second - small enough to iterate, which is the entire point of Phase 1.

Vectors are L2-normalised, so **inner product = cosine similarity** and FAISS's fast IP search
*is* cosine search. Results cache to `data/embeddings/{index_id}.npy`.

In [ ]:
from fitment_rag.embedding import Embedder

embedder = Embedder(cfg.embedding)
print(f"model={cfg.embedding.model}  dim={embedder.dim}  device={embedder.device}")

vectors, secs = embedder.embed_corpus([c["text"] for c in chunks], cfg.index_id)
print(f"matrix {vectors.shape}  {vectors.nbytes/1e6:.1f} MB  "
      f"({'from cache' if secs == 0 else f'{secs:.1f}s'})")

In [ ]:
# Sanity check: does this embedding space know anything about auto parts?
import pandas as pd

probe = ["brake pads for a Honda Civic", "ceramic disc brake pad set",
         "synthetic motor oil 5W-30", "leather steering wheel cover"]
v = embedder.encode(probe, is_query=True)

display(pd.DataFrame(v @ v.T, index=probe, columns=[p[:20] for p in probe]).round(3))

Rows 1 and 2 should sit clearly closer to each other than either does to motor oil. If they
do not, this embedder is wrong for the domain - and that is a **Phase 1a finding worth reporting**,
not something to paper over with a bigger LLM.

## Stage 4 - Vector index

At 2,400 vectors a brute-force scan is instant, so `faiss_flat` returns **exact** nearest
neighbours. That is what you want as the accuracy baseline: any approximate index is measured as
a loss against it.

HNSW and IVF trade recall for speed and only start paying for themselves in the tens of thousands
of vectors. That comparison is Phase 2 - here, exact is free.

In [ ]:
from fitment_rag.vectorstores.registry import build_store

store = build_store(cfg.vectorstore.backend, embedder.dim)
store.build(vectors, [c["chunk_id"] for c in chunks])
print(f"{store.name}: {len(store.ids):,} vectors in {store.build_seconds:.2f}s")

## Stage 5 - Retrieval

Three modes, and the third exists to keep the first two honest:

- **dense** - embed the query, take nearest neighbours. Strong on paraphrase, weak on exact
  alphanumeric part numbers.
- **bm25** - keyword scoring. Excellent on part numbers, blind to synonyms.
- **hybrid** - reciprocal rank fusion over both ranked lists.

Auto parts are a domain *full* of part numbers, so do not assume dense wins. Finding out is the
job of Phase 1c.

In [ ]:
from fitment_rag.retrieval import Retriever

retriever = Retriever(cfg.retrieval, chunks, embedder=embedder, store=store)

question = "Which brand makes ceramic brake pads?"
ids, scores, secs = retriever.retrieve([question])

print(f"query: {question}    ({secs*1000:.0f} ms)\n")
for rank, (cid, s) in enumerate(zip(ids[0], scores[0]), 1):
    print(f"{rank}. [{s:.3f}] {cid}")
    print(f"   {retriever.by_id[cid]['text'][:150]}...\n")

## Stage 6 - Scoring

Two families of metric, and the split between them is the whole design.

**Retrieval metrics** - did the right *document* reach the top-k? Scored at document level so a
config using 256-char chunks stays comparable to one using whole documents.

| metric | the question it answers |
|---|---|
| `hit@k` | was the right doc anywhere in the top k? |
| `recall@k` | what fraction of relevant docs were found? |
| `mrr` | how high did the first correct doc rank? |
| `ndcg@k` | rank-weighted quality of the whole list |

**Answer metrics** - deterministic string measures from `metrics/answer.py`. An LLM judge on a
15W CPU costs hours per sweep, and a 1.5B judge is not trustworthy enough to justify it.

In [ ]:
from fitment_rag.evalset.build import load_eval_set
from fitment_rag.metrics.retrieval import score_query

evalset = load_eval_set("../evalsets/amazon_automotive_1k.jsonl")
print(f"{len(evalset)} eval questions\n")

for q in evalset[:3]:
    print(f"Q: {q['question']}")
    print(f"A: {q['ground_truth']}")
    print(f"   doc={q['relevant_doc_ids'][0]}  field={q['source_field']}\n")

In [ ]:
q = evalset[0]
ids, _, _ = retriever.retrieve([q["question"]])
m = score_query(ids[0], set(q["relevant_doc_ids"]), cfg.eval.ks)

print(f"Q: {q['question']}")
print(f"want: {q['relevant_doc_ids'][0]}")
print(f"got : {[c.split('::')[0] for c in ids[0]]}\n")
for k, val in m.items():
    print(f"  {k:14} {val:.3f}")

## The whole thing, in one call

Everything above is what `pipeline.run()` does, plus caching, timing, and writing
`results/{run_id}/` containing:

- `metrics.json` - the aggregate numbers
- `config.json` - the exact settings that produced them
- `records.jsonl` - **every** retrieved chunk and generated answer, so a reviewer can audit any
  single query rather than taking the average on trust

`run_id` is a hash of the config, so a results directory is a fingerprint of the settings behind
it. You cannot accidentally compare two runs that differed in a knob you forgot you changed.

In [ ]:
from fitment_rag.pipeline import run

art = run(cfg)
{k: v for k, v in art.metrics.items()
 if k.startswith(("recall@", "mrr", "ndcg@", "answer_"))}

---

## What you have, and what you still do not know

You have one number. You do not know whether it is **good**, because there is nothing to compare
it to. Supplying that comparison is the entire job of notebook 02: hold everything fixed, vary
exactly one knob, and see which knobs actually move the metric.

Before you run it, write down your prediction. My honest guess: chunking and retrieval mode will
matter more than which embedding model you pick, and BM25 will land closer to dense than you
expect. A benchmark you predicted is worth ten you merely ran - and a writeup that says
"I expected X, got Y" is what separates research from a leaderboard screenshot.

**Next:** `02_retrieval_experiments.ipynb`